# 02 — Dynamic Retrieval

*Level 4 — Adaptive RAG*

## Objective
Let Top-K — and the retrieval strategy itself — vary with the classified complexity, instead of always running the same fixed pipeline regardless of how much evidence a question actually needs.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "query-classification", "dynamic-retrieval", "multi-hop-rag"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from dynamic_top_k import TOP_K_BY_COMPLEXITY, dynamic_top_k

for complexity, k in TOP_K_BY_COMPLEXITY.items():
    print(f"{complexity:<10} -> top_k={k}")


none       -> top_k=0
simple     -> top_k=3
complex    -> top_k=8
multi_hop  -> top_k=5


## The full decision policy on real questions


In [3]:
from adaptive_common.dataset import prepare
from adaptive_common.retrieval import DenseRetriever
from retrieval_policy import run_policy

data = prepare()
retriever = DenseRetriever.from_corpus(data.corpus)

questions = [
    "hi there!",
    "Where is Russell Hobbs based?",
    next(q["question"] for q in data.questions.values() if q["type"] == "comparison"),
    next(q["question"] for q in data.questions.values() if q["type"] == "bridge"),
]
for q in questions:
    result = run_policy(q, retriever)
    print(f"Q: {q[:70]}")
    print(f"  complexity={result['complexity']}  strategy={result['strategy']}  top_k={result['top_k']}  n_results={len(result['results'])}")
    print()


Q: hi there!
  complexity=none  strategy=no_retrieval  top_k=0  n_results=0



Q: Where is Russell Hobbs based?
  complexity=simple  strategy=single_retrieval  top_k=3  n_results=3



Q: Which airport is located in Maine, Sacramento International Airport or
  complexity=complex  strategy=multi_query_fusion  top_k=8  n_results=8



Q: Peter Hobbs founded the company that is based in what town in Manchest
  complexity=multi_hop  strategy=multi_hop_retrieval  top_k=5  n_results=7



## What I observed

A greeting correctly skips retrieval entirely (`top_k=0`, no search call at all — not just an empty result from a wasted search). A comparison question automatically gets a wider net (`top_k=8`, multi-query fusion) than a simple lookup (`top_k=3`, single retrieval) — the same corpus and retriever, but a genuinely different amount of work per question, decided automatically from the classification.

## Next

[03 — Corrective RAG](./03_corrective_rag.ipynb)
